# SD1.5 Out-of-Range CFG Recovery Ablation

This notebook analyzes the corrected recovery-only CFG ablation. It uses the same seven sampling ratios, five trials per setting, and optimization parameters as the corresponding main experiment while comparing unconditioned recovery and sunset-beach recovery at CFG 1, 1.5, 3, 5, and 7.5. The unconditioned and CFG 7.5 rows are synchronized from compatible main-experiment runs; CFG 1, 1.5, 3, and 5 are new recovery runs.

Run it top to bottom after any available first4/last3 ablation outputs exist under `results/`.
The analysis also ingests uniform MCS and pure inverse-square sampling when those baseline rows are available.


In [ ]:
from pathlib import Path
import importlib
import os
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
for search_root in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (search_root / 'ktilde' / 'unweighted' / 'config.json').is_file() and (search_root / 'src').is_dir():
        SD15_ROOT = search_root
        break
else:
    raise FileNotFoundError('Could not find sd1.5 project root.')

os.environ.setdefault('MPLCONFIGDIR', str(SD15_ROOT / 'results' / 'unweighted' / 'ablation' / 'out_of_range' / 'sunset' / 'figures' / '.matplotlib'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

if str(SD15_ROOT) not in sys.path:
    sys.path.insert(0, str(SD15_ROOT))
analysis_dir = SD15_ROOT / 'analyze_results'
if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

import sd15_cfg_ablation_analysis as cfgviz
import sd15_recovery_analysis as recovery
importlib.reload(cfgviz)
importlib.reload(recovery)

EXPERIMENT = 'out_of_range'
OUTPUT_DIR = SD15_ROOT / 'results' / 'unweighted' / 'ablation' / 'out_of_range' / 'sunset' / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SD15_ROOT, OUTPUT_DIR

## Load Results

This cell synchronizes compatible unconditioned and CFG 7.5 reference artifacts from the corresponding main first4/last3 outputs, then combines all available split ablation rows. The count table should reach five repeats for every sampling distribution, recovery line, and sampling ratio.

In [ ]:
LPIPS_TABLE = recovery.ensure_lpips_metrics(
    SD15_ROOT,
    result_namespace='unweighted',
    artifact_roots=[
        SD15_ROOT / 'results' / 'unweighted' / 'out_of_range' / 'sunset',
        SD15_ROOT / 'results' / 'unweighted' / 'ablation' / 'out_of_range' / 'sunset',
    ],
    device='cpu',
)

sync_report = cfgviz.sync_main_references(SD15_ROOT, experiment=EXPERIMENT)
display(sync_report.groupby(['distribution_key', 'status'], dropna=False).size().reset_index(name='copy_events'))

rows = cfgviz.load_cfg_ablation_rows(SD15_ROOT, experiment=EXPERIMENT)
print(f'Loaded {len(rows)} reconstruction rows')
display(cfgviz.count_table(rows))
display(rows[['distribution_key', 'line_condition', 'samp_perc', 'repeat_id', 'psnr_db', 'ssim', 'lpips', 'pixel_mae', 'case_root']].head())

## Metric Curves

This cell plots LPIPS versus sampling ratio for each sampling distribution. Each subplot compares unconditioned recovery with CFG 1, 1.5, 3, 5, and 7.5 recovery, plus the zero-filled baseline when available. Shading shows 95% confidence intervals across the five trials.

In [ ]:
paper_rows = rows[rows['distribution_key'].astype(str).str.startswith('k')].copy()
metric_outputs = cfgviz.plot_metric_curves(paper_rows, output_dir=OUTPUT_DIR, show=True, band='ci', metrics=('lpips',))
metric_outputs